In [ ]:
!pip install -q datasets scikit-learn

from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from collections import Counter
import json, re, numpy as np

print("Done.")

Done.


In [ ]:
dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

politeness_map = {
    "neutral":"neutral", "polite":"polite", "informal":"informal",
    "professional":"professional", "blunt":"blunt", "rude":"rude",
    "very_polite":"polite", "friendly":"informal", "religious":"polite",
    "impolite":"rude", "stern":"blunt", "sarcastic":"rude",
    "polite_but_firm":"polite"
}

positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal_t = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impolite","Rude",
            "Blunt","Crude","Envious","Dangerous","Fed up","Dry","Indifferent"}
emotional= {"Urgent","Pleading","Dramatic","Sad","Distressed","Tired",
            "Exhausted","Panicked","Shocked","Surprised","Concerned",
            "Apologetic","Appologetic","Regretful","Emotional","Teasing",
            "Casual","Dazed","Confused","Alarmed","Overwhelmed","Startled",
            "Devastated","Worried","Nervous/Sincere","Sad/Soft","Sad/Sweet",
            "Distress","Painful","Sorrowful","Weak","Uncertain","Weary",
            "Hurry","Impatient","Relieved","Fearful","Scared","Frightened",
            "Grieved","Incredulous","Bored","Lazy","Sleepy","Stressed",
            "Exasperated","Frustrated","Mischievous","Suggestive","Spooky",
            "Whispering","Indirect","Exclamatory","Lazy/Neutral",
            "Casual/Sleepy","Cheerful/Tired","Sweet/Weary","Sad/Serious",
            "Soft/Vulnerable","Natural","Considerate","Faithful","Trusting"}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal_t:  return "formal"
    if t in negative:  return "negative"
    if t in emotional: return "emotional"
    return "neutral"

power_map = {
    "equal":"equal",
    "inferior_to_superior":"inferior_to_superior",
    "superior_to_inferior":"superior_to_inferior",
    "any":"any",
    "customer_to_staff":"inferior_to_superior",
    "customer_to_seller":"inferior_to_superior",
    "passenger_to_driver":"inferior_to_superior",
    "patient_to_doctor":"inferior_to_superior",
}

def apply_all_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"], "neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"], "equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_all_labels)

le_pol  = LabelEncoder().fit(train_ds["politeness_coarse"])
le_reg  = LabelEncoder().fit(train_ds["register_label"])
le_pow  = LabelEncoder().fit(train_ds["power_coarse"])
le_tone = LabelEncoder().fit(train_ds["tone_coarse"])

print("Politeness:", list(le_pol.classes_))
print("Register:  ", list(le_reg.classes_))
print("Power:     ", list(le_pow.classes_))
print("Tone:      ", list(le_tone.classes_))

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Politeness: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]
Register:   [np.str_('colloquial'), np.str_('formal'), np.str_('slang'), np.str_('standard')]
Power:      [np.str_('any'), np.str_('equal'), np.str_('inferior_to_superior'), np.str_('superior_to_inferior')]
Tone:       [np.str_('emotional'), np.str_('formal'), np.str_('negative'), np.str_('neutral'), np.str_('positive')]


In [ ]:
def encode_all_labels(example):
    example["label_pol"]  = int(le_pol.transform([example["politeness_coarse"]])[0])
    example["label_reg"]  = int(le_reg.transform([example["register_label"]])[0])
    example["label_pow"]  = int(le_pow.transform([example["power_coarse"]])[0])
    example["label_tone"] = int(le_tone.transform([example["tone_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_all_labels)

split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

def get_weights(data, label_col, n_classes):
    counts = Counter(data[label_col])
    total  = len(data)
    return torch.tensor(
        [total / (n_classes * counts[i]) for i in range(n_classes)],
        dtype=torch.float
    )

w_pol  = get_weights(train_data, "label_pol",  len(le_pol.classes_))
w_reg  = get_weights(train_data, "label_reg",  len(le_reg.classes_))
w_pow  = get_weights(train_data, "label_pow",  len(le_pow.classes_))
w_tone = get_weights(train_data, "label_tone", len(le_tone.classes_))

print("Weights pol: ", [round(x,3) for x in w_pol.tolist()])
print("Weights reg: ", [round(x,3) for x in w_reg.tolist()])
print("Weights pow: ", [round(x,3) for x in w_pow.tolist()])
print("Weights tone:", [round(x,3) for x in w_tone.tolist()])

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Train: 1540  Val: 330  Test: 330
Weights pol:  [5.238, 0.976, 0.318, 0.817, 3.889, 6.111]
Weights reg:  [0.364, 4.01, 3.565, 1.39]
Weights pow:  [3.377, 0.33, 2.567, 3.5]
Weights tone: [1.149, 1.257, 4.738, 0.487, 0.936]


In [ ]:
# ── Cell 4: Generate batch prompt ────────────────────────────────────────────

prompts_export = []
for i, example in enumerate(test_data):
    reg   = example["register"]
    power = power_map.get(example["power_relation"], "equal")
    tone  = coarsen_tone(example["tone"])
    true_label = le_pol.classes_[example["label_pol"]]

    prompts_export.append({
        "id": i,
        "true_label": true_label,
        "utterance": example["utterance"],
        "context": str(example["context"]).strip(),
        "instruction": str(example["instruction"]).strip(),
        "register": reg,
        "power": power,
        "tone": tone,
    })

# ── Build the single mega-prompt ──────────────────────────────────────────────
batch_prompt = """You are a Burmese language expert specializing in pragmatics and politeness.

For each utterance below, classify the politeness level.
Choose EXACTLY ONE label from: blunt, informal, neutral, polite, professional, rude

Definitions:
- neutral: neither polite nor impolite, factual or indifferent
- polite: respectful, considerate, formal courtesy
- informal: casual, friendly, relaxed register
- professional: formal workplace or official language
- blunt: direct, no softening, but not rude
- rude: impolite, offensive, disrespectful

Respond ONLY with a JSON array like:
[{"id": 0, "label": "neutral"}, {"id": 1, "label": "polite"}, ...]
No explanation. No markdown. Just the JSON array.\n\n"""

for item in prompts_export:
    batch_prompt += f"""ID {item['id']}:
Context: {item['context']}
Instruction: {item['instruction']}
Register: {item['register']}
Power: {item['power']}
Tone: {item['tone']}
Utterance: {item['utterance']}

"""

batch_prompt += "Respond with the JSON array of all 330 predictions now."

print(f"Prompt length: {len(batch_prompt)} chars, ~{len(batch_prompt)//4} tokens")

# Save both files
with open("batch_prompt.txt", "w", encoding="utf-8") as f:
    f.write(batch_prompt)

with open("test_prompts.json", "w", encoding="utf-8") as f:
    json.dump(prompts_export, f, ensure_ascii=False, indent=2)

print("Saved batch_prompt.txt and test_prompts.json")

Prompt length: 55500 chars, ~13875 tokens
Saved batch_prompt.txt and test_prompts.json


In [19]:
# ── Split into 3 batches of 110 ───────────────────────────────────────────────

def make_batch_prompt(items):
    prompt = """You are a Burmese language expert specializing in pragmatics and politeness.

For each utterance below, classify the politeness level.
Choose EXACTLY ONE label from: blunt, informal, neutral, polite, professional, rude

Definitions:
- neutral: neither polite nor impolite, factual or indifferent
- polite: respectful, considerate, formal courtesy
- informal: casual, friendly, relaxed register
- professional: formal workplace or official language
- blunt: direct, no softening, but not rude
- rude: impolite, offensive, disrespectful

Respond ONLY with a JSON array like:
[{"id": 0, "label": "neutral"}, {"id": 1, "label": "polite"}, ...]
No explanation. No markdown. Just the raw JSON array.\n\n"""

    for item in items:
        prompt += f"""ID {item['id']}:
Context: {item['context']}
Instruction: {item['instruction']}
Register: {item['register']}
Power: {item['power']}
Tone: {item['tone']}
Utterance: {item['utterance']}

"""
    prompt += f"Respond with JSON array for all {len(items)} items above."
    return prompt

# Split into 3 batches
batch1 = prompts_export[0:110]
batch2 = prompts_export[110:220]
batch3 = prompts_export[220:330]

for i, batch in enumerate([batch1, batch2, batch3], 1):
    with open(f"batch_{i}.txt", "w", encoding="utf-8") as f:
        f.write(make_batch_prompt(batch))
    print(f"batch_{i}.txt — IDs {batch[0]['id']} to {batch[-1]['id']}, "
          f"~{len(make_batch_prompt(batch))//4} tokens")

batch_1.txt — IDs 0 to 109, ~4716 tokens
batch_2.txt — IDs 110 to 219, ~4719 tokens
batch_3.txt — IDs 220 to 329, ~4809 tokens


In [20]:
import json

# Load from file instead of pasting
with open("chatgpt_predictions.json") as f:
    chatgpt_results = json.load(f)

with open("gemini_predictions.json") as f:
    gemini_results = json.load(f)

# Score ChatGPT
MODEL_NAME_EVAL = "ChatGPT o4-mini"
pred_dict = {r["id"]: r["label"] for r in chatgpt_results}
predictions_chat = [pred_dict[i] for i in range(330)]
true_names = [le_pol.classes_[example["label_pol"]] for example in test_data]

# then run the scoring block...

In [21]:
# ── Score block — run once per model ─────────────────────────────────────────

import json
from sklearn.metrics import classification_report, accuracy_score, f1_score

CLASSES = list(le_pol.classes_)
true_names = [le_pol.classes_[example["label_pol"]] for example in test_data]

# ── Change these two lines per model ─────────────────────────────────────────
MODEL_NAME_EVAL = "Gemini 3 Fast"
pred_file       = "gemini_predictions.json"
# ─────────────────────────────────────────────────────────────────────────────

with open(pred_file) as f:
    results = json.load(f)

pred_dict = {r["id"]: r["label"].strip().lower() for r in results}
missing = [i for i in range(330) if i not in pred_dict]
if missing:
    print(f"Warning: {len(missing)} missing IDs defaulted to neutral: {missing}")

predictions = [pred_dict.get(i, "neutral") for i in range(len(test_data))]

acc      = accuracy_score(true_names, predictions)
macro_f1 = f1_score(true_names, predictions, average="macro",
                    labels=CLASSES, zero_division=0)
w_f1     = f1_score(true_names, predictions, average="weighted",
                    labels=CLASSES, zero_division=0)

print(f"\n── {MODEL_NAME_EVAL} Zero-Shot Results ──────────────────────────")
print(f"Accuracy:    {acc:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {w_f1:.4f}")
print()
print(classification_report(true_names, predictions, labels=CLASSES, zero_division=0))

print("\n── Comparison ──────────────────────────────────────────────────────")
print(f"{'Model':<40} {'Macro F1':>10}")
print("-" * 52)
print(f"{'Stage 1 (utterance only)':<40} {'0.7014':>10}")
print(f"{'Stage 2 (+ context)':<40} {'0.7583':>10}")
print(f"{'Stage 3 (oracle metadata)':<40} {'0.7990':>10}")
print(f"{'Stage 4 (predicted metadata)':<40} {'0.7020':>10}")
print(f"{MODEL_NAME_EVAL + ' (zero-shot)':<40} {macro_f1:>10.4f}")

# ── Save results ──────────────────────────────────────────────────────────────
output = []
for i, (pred, true) in enumerate(zip(predictions, true_names)):
    output.append({
        "id": i,
        "true_label": true,
        "predicted_label": pred,
        "correct": pred == true
    })

fname = MODEL_NAME_EVAL.replace(" ", "_").replace("-", "_")
with open(f"results_{fname}.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

with open(f"summary_{fname}.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME_EVAL,
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(w_f1, 4),
        "n_examples": 330,
        "n_missing": len(missing),
    }, f, indent=2)

print(f"\nSaved results_{fname}.json and summary_{fname}.json")


── Gemini 3 Fast Zero-Shot Results ──────────────────────────
Accuracy:    0.5000
Macro F1:    0.4929
Weighted F1: 0.5021

              precision    recall  f1-score   support

       blunt       0.29      0.71      0.42        14
    informal       0.38      0.78      0.51        59
     neutral       0.73      0.43      0.54       174
      polite       0.45      0.30      0.36        56
professional       0.41      0.73      0.52        15
        rude       0.75      0.50      0.60        12

    accuracy                           0.50       330
   macro avg       0.50      0.58      0.49       330
weighted avg       0.59      0.50      0.50       330


── Comparison ──────────────────────────────────────────────────────
Model                                      Macro F1
----------------------------------------------------
Stage 1 (utterance only)                     0.7014
Stage 2 (+ context)                          0.7583
Stage 3 (oracle metadata)                    0.7990
St

In [22]:
# ── Score block — run once per model ─────────────────────────────────────────

import json
from sklearn.metrics import classification_report, accuracy_score, f1_score

CLASSES = list(le_pol.classes_)
true_names = [le_pol.classes_[example["label_pol"]] for example in test_data]

# ── Change these two lines per model ─────────────────────────────────────────
MODEL_NAME_EVAL = "ChatGPT o4-mini"           # change per model
pred_file       = "chatgpt_predictions.json"  # change per model
# ─────────────────────────────────────────────────────────────────────────────

with open(pred_file) as f:
    results = json.load(f)

pred_dict = {r["id"]: r["label"].strip().lower() for r in results}
missing = [i for i in range(330) if i not in pred_dict]
if missing:
    print(f"Warning: {len(missing)} missing IDs defaulted to neutral: {missing}")

predictions = [pred_dict.get(i, "neutral") for i in range(len(test_data))]

acc      = accuracy_score(true_names, predictions)
macro_f1 = f1_score(true_names, predictions, average="macro",
                    labels=CLASSES, zero_division=0)
w_f1     = f1_score(true_names, predictions, average="weighted",
                    labels=CLASSES, zero_division=0)

print(f"\n── {MODEL_NAME_EVAL} Zero-Shot Results ──────────────────────────")
print(f"Accuracy:    {acc:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {w_f1:.4f}")
print()
print(classification_report(true_names, predictions, labels=CLASSES, zero_division=0))

print("\n── Comparison ──────────────────────────────────────────────────────")
print(f"{'Model':<40} {'Macro F1':>10}")
print("-" * 52)
print(f"{'Stage 1 (utterance only)':<40} {'0.7014':>10}")
print(f"{'Stage 2 (+ context)':<40} {'0.7583':>10}")
print(f"{'Stage 3 (oracle metadata)':<40} {'0.7990':>10}")
print(f"{'Stage 4 (predicted metadata)':<40} {'0.7020':>10}")
print(f"{MODEL_NAME_EVAL + ' (zero-shot)':<40} {macro_f1:>10.4f}")

# ── Save results ──────────────────────────────────────────────────────────────
output = []
for i, (pred, true) in enumerate(zip(predictions, true_names)):
    output.append({
        "id": i,
        "true_label": true,
        "predicted_label": pred,
        "correct": pred == true
    })

fname = MODEL_NAME_EVAL.replace(" ", "_").replace("-", "_")
with open(f"results_{fname}.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

with open(f"summary_{fname}.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME_EVAL,
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(w_f1, 4),
        "n_examples": 330,
        "n_missing": len(missing),
    }, f, indent=2)

print(f"\nSaved results_{fname}.json and summary_{fname}.json")


── ChatGPT o4-mini Zero-Shot Results ──────────────────────────
Accuracy:    0.5636
Macro F1:    0.6346
Weighted F1: 0.5372

              precision    recall  f1-score   support

       blunt       0.32      0.86      0.47        14
    informal       0.43      0.95      0.59        59
     neutral       1.00      0.29      0.45       174
      polite       0.55      0.77      0.64        56
professional       0.65      1.00      0.79        15
        rude       0.91      0.83      0.87        12

    accuracy                           0.56       330
   macro avg       0.64      0.78      0.63       330
weighted avg       0.77      0.56      0.54       330


── Comparison ──────────────────────────────────────────────────────
Model                                      Macro F1
----------------------------------------------------
Stage 1 (utterance only)                     0.7014
Stage 2 (+ context)                          0.7583
Stage 3 (oracle metadata)                    0.7990
